# Telematics DLT Lab - Driving Data Generation

In this notebook, we'll demonstrate how you can use the `insurance_telematics` package to simulate realistic driving journies, and score drivers transparently on a range of metrics using open-source machine learning models.


In [1]:
from insurance_telematics import (
    load_trips, 
    clean_trips, 
    extract_trip_features, 
    aggregate_to_driver,
    TripSimulator, 
    TelematicsScoringPipeline,
)
import plotly.express as px
import pandas as pd

In [2]:
sim = TripSimulator(seed=42)
trips_df, claims_df = sim.simulate(n_drivers=100, trips_per_driver=50)

Save data to a `.csv` to costly generations step.

In [13]:
# Note, these are polars DataFrames
trips_df.write_csv("data/trips.csv")
claims_df.write_csv("data/claims.csv")

In [3]:
trips_df.head()

driver_id,trip_id,timestamp,latitude,longitude,speed_kmh,acceleration_ms2,heading_deg
str,str,"datetime[μs, UTC]",f64,f64,f64,f64,f64
"""DRV0000""","""TRP0000000""",2024-01-02 03:47:42.816032 UTC,51.226474,-0.096828,30.58,-0.3422,356.4
"""DRV0000""","""TRP0000000""",2024-01-02 03:47:43.816032 UTC,51.226555,-0.096837,32.59,0.4068,355.0
"""DRV0000""","""TRP0000000""",2024-01-02 03:47:44.816032 UTC,51.226634,-0.096848,31.91,-0.4036,354.1
"""DRV0000""","""TRP0000000""",2024-01-02 03:47:45.816032 UTC,51.226715,-0.096861,32.73,0.1675,353.6
"""DRV0000""","""TRP0000000""",2024-01-02 03:47:46.816032 UTC,51.226797,-0.096876,33.01,-0.0207,354.9


In [4]:
trips_df.describe()

statistic,driver_id,trip_id,timestamp,latitude,longitude,speed_kmh,acceleration_ms2,heading_deg
str,str,str,str,f64,f64,f64,f64,f64
"""count""","""9840849""","""9840849""","""9840849""",9.840849e6,9.840849e6,9.840849e6,9.840849e6,9.840849e6
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,"""2024-01-25 19:41:50.587807+00:…",51.497297,-0.089535,45.461896,0.003796,179.301282
"""std""",null,null,null,0.310922,0.328127,11.595277,1.439247,103.69276
"""min""","""DRV0000""","""TRP0000000""","""2024-01-01 15:42:52.346833+00:…",50.67403,-1.296234,0.0,-12.7918,0.0
"""25%""",null,null,"""2024-01-13 23:08:01.476577+00:…",51.242661,-0.343454,37.52,-0.8988,90.5
"""50%""",null,null,"""2024-01-25 21:00:49.030173+00:…",51.499044,-0.08787,44.17,-0.1044,178.0
"""75%""",null,null,"""2024-02-07 00:20:18.931278+00:…",51.754203,0.171793,52.04,0.769,268.8
"""max""","""DRV0099""","""TRP0004999""","""2024-02-19 01:27:57.952361+00:…",52.378587,0.960904,149.31,13.1719,360.0


### Visualise the trips on a map

In [ ]:
fig = px.line_geo(
    trips_df[:1000],
    lat="latitude",
    lon="longitude",
    color="trip_id",
    line_group="trip_id",
    title="Visualising Simulated Trips",
)
fig.show(renderer="browser")

KeyboardInterrupt: 

In [6]:
trips_df.head(), claims_df.head()

(shape: (5, 8)
 ┌───────────┬────────────┬────────────┬───────────┬───────────┬───────────┬────────────┬───────────┐
 │ driver_id ┆ trip_id    ┆ timestamp  ┆ latitude  ┆ longitude ┆ speed_kmh ┆ accelerati ┆ heading_d │
 │ ---       ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ on_ms2     ┆ eg        │
 │ str       ┆ str        ┆ datetime[μ ┆ f64       ┆ f64       ┆ f64       ┆ ---        ┆ ---       │
 │           ┆            ┆ s, UTC]    ┆           ┆           ┆           ┆ f64        ┆ f64       │
 ╞═══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪════════════╪═══════════╡
 │ DRV0000   ┆ TRP0000000 ┆ 2024-01-02 ┆ 51.226474 ┆ -0.096828 ┆ 30.58     ┆ -0.3422    ┆ 356.4     │
 │           ┆            ┆ 03:47:42.8 ┆           ┆           ┆           ┆            ┆           │
 │           ┆            ┆ 16032 UTC  ┆           ┆           ┆           ┆            ┆           │
 │ DRV0000   ┆ TRP0000000 ┆ 2024-01-02 ┆ 51.226555 ┆ -0.096837 ┆ 32

## Making Predictions

In [7]:
pipe = TelematicsScoringPipeline(n_hmm_states=3)
pipe.fit(trips_df, claims_df)
predictions = pipe.predict(trips_df)

c:\dev\Misc\dlt-lab-data\.venv\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\dev\Misc\dlt-lab-data\.venv\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\dev\Misc\dlt-lab-data\.venv\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\dev\Misc\dlt-lab-data\.venv\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\dev\Misc\dlt-lab-data\.venv\lib\site-packages\statsmodels\genmod\generalized_linear_m